In [1]:
import cv2
import numpy as np
import glob
import yaml
import pickle
import tkinter as tk
from tkinter import simpledialog

In [2]:
with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)["field_rotation"]

In [3]:
frame_files = sorted(glob.glob(config["input"]))
homography = []
image_path = frame_files[0]
output_path = config["output"] + image_path[-9:-4]

In [4]:
 # Load the image
image = cv2.imread(image_path)

# Store clicked pixel points and corresponding field positions
pixel_points = []
real_world_points = []

# Initialize Tkinter root window (for input pop-up)
root = tk.Tk()
root.withdraw()  # Hide the root window

# Mouse click callback function
def get_points(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"Clicked at: ({x}, {y})")
        pixel_points.append((x, y))

        # Pop-up to enter real-world coordinates
        real_x = simpledialog.askfloat("Input", f"Enter real-world X coordinate for ({x}, {y}):")
        real_y = simpledialog.askfloat("Input", f"Enter real-world Y coordinate for ({x}, {y}):")

        # Store the entered values
        if real_x is not None and real_y is not None:
            real_world_points.append((real_x, real_y))
            print(f"Mapped to: ({real_x}, {real_y}) meters")

# Display the image and capture points
cv2.namedWindow("Select Points", cv2.WINDOW_NORMAL)
cv2.setMouseCallback("Select Points", get_points)

while True:
    cv2.imshow("Select Points", image)
    key = cv2.waitKey(1) & 0xFF
    if key == ord("q") and len(pixel_points) >= 4:  # Press 'q' to finish
        break

cv2.destroyAllWindows()

# Convert lists to NumPy arrays
pixel_points = np.array(pixel_points, dtype=np.float32)
real_world_points = np.array(real_world_points, dtype=np.float32)

# Save the collected data
print("\nFinal Mappings:")
for i in range(len(pixel_points)):
    print(f"Pixel {pixel_points[i]} -> Real-World {real_world_points[i]}")


Clicked at: (173, 930)
Mapped to: (22.0, 0.0) meters
Clicked at: (697, 390)
Mapped to: (0.0, 70.0) meters
Clicked at: (1293, 413)
Mapped to: (22.0, 70.0) meters
Clicked at: (1542, 798)
Mapped to: (40.0, 15.0) meters

Final Mappings:
Pixel [173. 930.] -> Real-World [22.  0.]
Pixel [697. 390.] -> Real-World [ 0. 70.]
Pixel [1293.  413.] -> Real-World [22. 70.]
Pixel [1542.  798.] -> Real-World [40. 15.]


In [5]:
# Compute the homography matrix
homography_matrix, _ = cv2.findHomography(pixel_points, real_world_points)

# Function to convert pixel coordinates to real-world coordinates
def pixel_to_real(px, py, H):
    pixel_coord = np.array([[px, py, 1]], dtype=np.float32).T
    real_coord = np.dot(H, pixel_coord)
    real_coord /= real_coord[2]  # Normalize
    return real_coord[0][0], real_coord[1][0]  # X, Y in meters

# Example test: Convert one of the field points to real-world coordinates
px, py = pixel_points[0]
real_x, real_y = pixel_to_real(px, py, homography_matrix)
print(f"Example: Pixel ({px}, {py}) -> Real-world ({real_x:.2f}m, {real_y:.2f}m)")


Example: Pixel (173.0, 930.0) -> Real-world (22.00m, 0.00m)


In [7]:
with open(output_path,'wb') as f:
    pickle.dump(homography_matrix, f)
